# Step 4 — Sentence embeddings (Stage A: pretrained, no training)

**Input:** `outputs/equal_step2_masking/step2_masked_analysis_text.csv` (cased, with line breaks and bullet markers still present).
**Output:** sentence units and their SentenceTransformers embeddings, plus an exploratory comparison with a TF-IDF baseline.

Sentences are segmented from the Step 2 masked text rather than the Step 3 text, because Step 3 collapses the line breaks and bullet markers that mark sentence boundaries, and lowercasing is unnecessary for the embedding model. Step 3 text remains for lexical methods.

No model is trained here. Clusters are exploratory and are not bias judgments.

In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import adjusted_rand_score, silhouette_score

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 200)

INPUT_PATH = Path("outputs") / "equal_step2_masking" / "step2_masked_analysis_text.csv"
OUTPUT_DIR = Path("outputs") / "equal_step4_sentence_embeddings"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Multilingual because the advertisements may contain Filipino/Taglish wording.
MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"
N_CLUSTERS = 6
RANDOM_STATE = 42

data = pd.read_csv(INPUT_PATH, encoding="utf-8-sig", dtype=str, keep_default_na=False)
print(f"{len(data):,} advertisements loaded.")

## 1. Segmentation

Each text component is split by line; leading bullet markers are removed; lines are split further at `.`, `!` or `?` followed by whitespace. Units without any letter are dropped. Case and punctuation are kept. The job title is kept as its own unit (`component = job_title`).

In [ ]:
BULLET_PREFIX_RE = re.compile(r"^\s*(?:[•●▪▫◦‣⁃○◉■□◆◇➢➤✓✔]+|[-*](?=\s|$))\s*")
SPLIT_RE = re.compile(r"(?<=[.!?])\s+")
COMPONENTS = {
    "masked_job_title": "job_title",
    "masked_body_other_text": "body_other",
    "masked_qualification_text": "qualification",
}


def segment(text):
    for line in text.splitlines():
        line = BULLET_PREFIX_RE.sub("", line, count=1)
        for unit in SPLIT_RE.split(line):
            unit = re.sub(r"\s+", " ", unit).strip()
            if re.search(r"[^\W\d_]", unit):
                yield unit


assert list(segment("• Barista, Mechanic.  Must lead! \n- Ok")) == ["Barista, Mechanic.", "Must lead!", "Ok"]

rows = []
for record in data.itertuples(index=False):
    position = 0
    for column, component in COMPONENTS.items():
        for unit in segment(getattr(record, column)):
            position += 1
            rows.append(
                {
                    "sentence_id": f"{record.record_id}-s{position:03d}",
                    "record_id": record.record_id,
                    "employer_id": record.employer_id,
                    "component": component,
                    "sentence": unit,
                }
            )
sentences = pd.DataFrame(rows)
print(f"{len(sentences):,} sentence units from {sentences['record_id'].nunique():,} advertisements.")
print(sentences["component"].value_counts().to_string())
print(f"Exact-duplicate sentence texts: {sentences['sentence'].duplicated().sum():,}")
display(sentences.sample(8, random_state=RANDOM_STATE))

## 2. Embeddings

Identical sentence strings are encoded once and mapped back, so repeated template lines receive identical vectors. Vectors are L2-normalized, so dot product equals cosine similarity.

In [ ]:
model = SentenceTransformer(MODEL_NAME)
unique_text, inverse = np.unique(sentences["sentence"].to_numpy(), return_inverse=True)
unique_vectors = model.encode(
    unique_text.tolist(), batch_size=64, normalize_embeddings=True, show_progress_bar=False, convert_to_numpy=True
)
embeddings = unique_vectors[inverse].astype("float32")
assert embeddings.shape == (len(sentences), model.get_sentence_embedding_dimension())
assert np.allclose(np.linalg.norm(embeddings, axis=1), 1.0, atol=1e-4)
print(f"Model: {MODEL_NAME}; embedding matrix: {embeddings.shape}; unique texts encoded: {len(unique_text):,}")

np.save(OUTPUT_DIR / "step4_sentence_embeddings.npy", embeddings)
sentences.to_csv(OUTPUT_DIR / "step4_sentence_units.csv", index=False, encoding="utf-8-sig")

## 3. Sanity check: nearest neighbours

For a few sentences containing occupational or gendered wording, the most similar other (distinct) sentences are shown. This is a qualitative check that the space reflects meaning.

In [ ]:
probe_pattern = r"\b(?:female|male|barista|mechanic|salesman|saleswoman|young|aggressive|strong|supportive)\b"
probes = sentences[sentences["sentence"].str.contains(probe_pattern, case=False, regex=True)]
probes = probes.drop_duplicates("sentence").sample(min(4, len(probes)), random_state=RANDOM_STATE)
for idx in probes.index:
    scores = embeddings @ embeddings[idx]
    order = [j for j in np.argsort(-scores) if sentences.at[j, "sentence"] != sentences.at[idx, "sentence"]][:3]
    print(f"\nQUERY: {sentences.at[idx, 'sentence']}")
    for j in order:
        print(f"  {scores[j]:.3f}  {sentences.at[j, 'sentence']}")

## 4. Exploratory clustering and TF-IDF comparison

K-means with the same number of clusters is run on the embeddings and on TF-IDF features. The adjusted Rand index (ARI) measures how much the two partitions agree; low agreement is expected because they capture different structure. Silhouette values are computed on a random subsample and are not comparable across different feature spaces; they are shown only as a rough indication of separation.

In [ ]:
emb_km = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=10).fit(embeddings)
sentences["embedding_cluster"] = emb_km.labels_

tfidf = TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=3, max_df=0.5, sublinear_tf=True)
tfidf_matrix = tfidf.fit_transform(sentences["sentence"])
tfidf_km = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=10).fit(tfidf_matrix)
sentences["tfidf_cluster"] = tfidf_km.labels_

rng = np.random.default_rng(RANDOM_STATE)
sub = rng.choice(len(sentences), size=min(5000, len(sentences)), replace=False)
print(f"ARI (embedding vs TF-IDF clusters): {adjusted_rand_score(sentences['embedding_cluster'], sentences['tfidf_cluster']):.3f}")
print(f"Silhouette, embeddings (cosine, n={len(sub):,}): {silhouette_score(embeddings[sub], emb_km.labels_[sub], metric='cosine'):.3f}")
print(f"Silhouette, TF-IDF (cosine, n={len(sub):,}): {silhouette_score(tfidf_matrix[sub], tfidf_km.labels_[sub], metric='cosine'):.3f}")

display(pd.crosstab(sentences["embedding_cluster"], sentences["component"], margins=True))
display(pd.crosstab(sentences["embedding_cluster"], sentences["tfidf_cluster"], margins=True))

In [ ]:
# Representative sentences: the units closest to each embedding-cluster centroid.
summary_rows = []
for cluster in range(N_CLUSTERS):
    members = np.flatnonzero(emb_km.labels_ == cluster)
    closest = members[np.argsort(-(embeddings[members] @ emb_km.cluster_centers_[cluster]))]
    reps = sentences.loc[closest, "sentence"].drop_duplicates().head(5).tolist()
    summary_rows.append(
        {
            "embedding_cluster": cluster,
            "units": len(members),
            "advertisements": sentences.loc[members, "record_id"].nunique(),
            "employers": sentences.loc[members, "employer_id"].nunique(),
            "representative_sentences": " || ".join(reps),
        }
    )
cluster_summary = pd.DataFrame(summary_rows)
for row in summary_rows:
    print(f"\nCluster {row['embedding_cluster']}: {row['units']:,} units, {row['advertisements']:,} ads, {row['employers']:,} employers")
    for text in row["representative_sentences"].split(" || "):
        print(f"   - {text}")

sentences.to_csv(OUTPUT_DIR / "step4_sentence_units.csv", index=False, encoding="utf-8-sig")
cluster_summary.to_csv(OUTPUT_DIR / "step4_embedding_cluster_summary.csv", index=False, encoding="utf-8-sig")
print("\nSaved:", *sorted(p.name for p in OUTPUT_DIR.iterdir()), sep="\n  ")

## Limitations

- The pretrained model carries its own social biases, and embedding similarity is not a bias judgment.
- Repeated employer template lines inflate some clusters; cluster-level employer counts above show how concentrated each cluster is.
- Sentence units are analysis segments, not independent observations; sentences from the same advertisement or employer are correlated.
- K-means with a fixed number of clusters is a convenience, not evidence of six natural groups.